In [ ]:
"""
Unit III — Problem 14: Transfer Learning Layer Freezing Strategy
================================================================
Research question: Which layers should you freeze when fine-tuning on a
                   small custom dataset like IndiCraft?

What this script does
---------------------
1. Fine-tunes ImageNet-pretrained ResNet-50 under 4 freezing strategies:
      A. Full fine-tune       (freeze nothing)
      B. Freeze early layers  (freeze layer1 + layer2)
      C. Head only            (freeze everything except fc)
      D. Freeze late layers   (freeze layer3 + layer4 — unusual hypothesis)
2. Tracks per epoch: train accuracy, val accuracy, wall-clock time.
3. Reports: overfitting gap (train − val), convergence speed, final test acc.
4. Breaks down final accuracy per class to show where freezing hurts minority classes.

Outputs
-------
results/unit3/
    freezing_strategy_curves.png    — val acc curves for all strategies
    overfitting_gap.png             — (train−val) gap per epoch per strategy
    per_class_breakdown.png         — per-class test acc bar chart
    training_summary.png            — table: acc / time / gap side by side
    results.json                    — all numbers
"""
"""
Unit III — Problem 14: Transfer Learning Layer Freezing Strategy
================================================================
Research question: Which layers should you freeze when fine-tuning on a
                   small custom dataset like IndiCraft?

What this script does
---------------------
1. Fine-tunes ImageNet-pretrained ResNet-50 under 4 freezing strategies:
      A. Full fine-tune       (freeze nothing)
      B. Freeze early layers  (freeze layer1 + layer2 + stem)
      C. Head only            (freeze everything except fc)
      D. Freeze late layers   (freeze layer3 + layer4)
2. Tracks per epoch: train accuracy, val accuracy, wall-clock time.
3. Early stopping with best-weight restoration.
4. Reports: overfitting gap, convergence speed, final test acc, per-class acc.

Outputs  →  results/unit3/
    freezing_strategy_curves.png
    overfitting_gap.png
    per_class_breakdown.png
    training_summary.png
    results.json
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json, time, random

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# ── CONFIG ────────────────────────────────────────────────────────────────────
CROPS_ROOT  = Path("/content/drive/MyDrive/dlcv_phase_2/IndiCraft-crops")
OUT_DIR     = Path("/content/drive/MyDrive/dlcv_phase_2/results/unit1")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS      = 5
BATCH       = 32
NUM_CLASSES = 8
PATIENCE    = 4

# Strategy-specific learning rates (head-only trains fewest params → higher LR)
LR_CONFIG = {
    "Full fine-tune":        1e-4,
    "Freeze early (L1+L2)":  3e-4,
    "Head only":             1e-3,
    "Freeze late (L3+L4)":   3e-4,
}

STRATEGIES = list(LR_CONFIG.keys())

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── TRANSFORMS ────────────────────────────────────────────────────────────────
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])

train_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    normalize,
])

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

# ── DATA ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_ds = datasets.ImageFolder(CROPS_ROOT / "train", transform=train_tfm)
    val_ds   = datasets.ImageFolder(CROPS_ROOT / "val",   transform=eval_tfm)
    test_ds  = datasets.ImageFolder(CROPS_ROOT / "test",  transform=eval_tfm)

    # Derive class names from folder structure — not hardcoded
    class_names = train_ds.classes
    print(f"Classes detected: {class_names}")

    # num_workers: 2 with GPU, 0 on CPU/Windows to avoid multiprocessing issues
    nw = 2 if torch.cuda.is_available() else 0

    return (
        DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=nw),
        DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=nw),
        DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=nw),
        class_names,
    )

# ── FREEZING STRATEGY ─────────────────────────────────────────────────────────
# Maps each strategy to the name substrings of layers to freeze
FREEZE_PREFIXES = {
    "Full fine-tune":        [],                                     # nothing frozen
    "Freeze early (L1+L2)":  ["conv1", "bn1", "layer1", "layer2"],  # stem + early
    "Head only":             None,                                   # handled separately
    "Freeze late (L3+L4)":   ["layer3", "layer4"],                  # deep layers
}

def apply_freeze_strategy(model: nn.Module, strategy: str) -> nn.Module:
    """
    Freeze parameters according to strategy, then lock BatchNorm running stats
    only for the layers whose parameters are actually frozen.
    """
    # Step 1 — unfreeze everything
    for p in model.parameters():
        p.requires_grad = True

    # Step 2 — freeze by strategy
    if strategy == "Head only":
        for name, p in model.named_parameters():
            if "fc" not in name:
                p.requires_grad = False
    else:
        prefixes = FREEZE_PREFIXES[strategy]
        for name, p in model.named_parameters():
            if any(x in name for x in prefixes):
                p.requires_grad = False

    # Step 3 — lock BN running stats ONLY for frozen layers
    # (avoids corrupting statistics of layers still being trained)
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            params_frozen = all(
                not p.requires_grad for p in module.parameters()
            )
            if params_frozen:
                module.eval()   # freeze running mean/var for this BN only

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)")
    return model

# ── EARLY STOPPING WITH BEST-WEIGHT RESTORE ───────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 4):
        self.patience     = patience
        self.best_acc     = 0.0
        self.counter      = 0
        self.best_weights = None  # saved state_dict at best val acc

    def step(self, val_acc: float, model: nn.Module) -> bool:
        """Returns True when training should stop."""
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter  = 0
            # Deep-copy weights to CPU to avoid GPU memory overhead
            self.best_weights = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

    def restore(self, model: nn.Module):
        """Load the best weights back into the model."""
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)

# ── TRAIN ONE EPOCH ───────────────────────────────────────────────────────────
def run_epoch(model: nn.Module, loader: DataLoader,
              criterion, optimizer=None):
    """
    Single train or eval pass.
    Uses torch.set_grad_enabled for clean gradient control.
    When optimizer is None → evaluation mode.
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    loss_sum = correct = total = 0

    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out  = model(x)
            loss = criterion(out, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            loss_sum += loss.item() * y.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)

    return loss_sum / total, correct / total

# ── PER-CLASS TEST EVALUATION ─────────────────────────────────────────────────
def evaluate_per_class(model: nn.Module, test_dl: DataLoader,
                       class_names: list) -> tuple:
    """Returns (overall_acc, {class_name: acc}) on the test set."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in test_dl:
            all_preds.append(model(x.to(DEVICE)).argmax(1).cpu())
            all_labels.append(y)

    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()

    per_class = {}
    for c, name in enumerate(class_names):
        mask = labels == c
        per_class[name] = (
            float((preds[mask] == c).mean()) if mask.sum() > 0 else 0.0
        )

    overall = float((preds == labels).mean())
    return overall, per_class

# ── MAIN TRAINING LOOP ────────────────────────────────────────────────────────
def train_all():
    train_dl, val_dl, test_dl, class_names = get_loaders()
    criterion = nn.CrossEntropyLoss()
    all_results = {}

    for strat in STRATEGIES:
        print(f"\n{'='*60}")
        print(f"Strategy: {strat}")

        # Fresh pretrained ResNet-50 for each strategy
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        model = apply_freeze_strategy(model, strat).to(DEVICE)

        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_CONFIG[strat],
            weight_decay=1e-4,       # L2 regularisation — helps small datasets
        )

        # Cosine annealing over full epoch budget
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS
        )

        stopper = EarlyStopping(PATIENCE)
        train_accs, val_accs, epoch_times = [], [], []

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()

            _, train_acc = run_epoch(model, train_dl, criterion, optimizer)
            _, val_acc   = run_epoch(model, val_dl,   criterion)

            scheduler.step()
            elapsed = time.time() - t0

            train_accs.append(train_acc)
            val_accs.append(val_acc)
            epoch_times.append(elapsed)

            print(f"  Ep {epoch:2d}  train={train_acc:.3f}  val={val_acc:.3f}"
                  f"  t={elapsed:.1f}s")

            if stopper.step(val_acc, model):
                print("  ↳ Early stopping triggered — restoring best weights")
                stopper.restore(model)
                break

        # Ensure best weights are loaded even if no early stop triggered
        stopper.restore(model)

        # Final test evaluation (per-class)
        test_acc, per_class = evaluate_per_class(model, test_dl, class_names)
        total_time          = sum(epoch_times)
        gap                 = [t - v for t, v in zip(train_accs, val_accs)]

        all_results[strat] = {
            "train_accs":            train_accs,
            "val_accs":              val_accs,
            "overfitting_gap":       gap,
            "test_acc":              test_acc,
            "per_class":             per_class,
            "total_training_time_s": total_time,
            "avg_epoch_time_s":      float(np.mean(epoch_times)),
        }
        print(f"  ✓ Test acc: {test_acc:.3f}  |  Total time: {total_time:.0f}s")

    return all_results, class_names

# ── PLOTS ─────────────────────────────────────────────────────────────────────
COLORS = plt.cm.Set1(np.linspace(0, 0.8, 4))

def plot_val_curves(results: dict):
    fig, ax = plt.subplots(figsize=(10, 5))
    for strat, color in zip(STRATEGIES, COLORS):
        epochs = range(1, len(results[strat]["val_accs"]) + 1)
        ax.plot(epochs, results[strat]["val_accs"],
                label=strat, color=color, linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Accuracy")
    ax.set_title("Validation Accuracy — Freezing Strategies (ResNet-50, IndiCraft)")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    path = OUT_DIR / "freezing_strategy_curves.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"Saved: {path}")

def plot_overfitting_gap(results: dict):
    fig, ax = plt.subplots(figsize=(10, 5))
    for strat, color in zip(STRATEGIES, COLORS):
        gap    = results[strat]["overfitting_gap"]
        epochs = range(1, len(gap) + 1)
        ax.plot(epochs, gap, label=strat, color=color, linewidth=2)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train − Val Accuracy")
    ax.set_title("Overfitting Gap per Strategy")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = OUT_DIR / "overfitting_gap.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"Saved: {path}")

def plot_per_class(results: dict, class_names: list):
    x     = np.arange(len(class_names))
    width = 0.2
    fig, ax = plt.subplots(figsize=(14, 5))
    for i, (strat, color) in enumerate(zip(STRATEGIES, COLORS)):
        vals = [results[strat]["per_class"].get(c, 0.0) for c in class_names]
        ax.bar(x + i * width, vals, width,
               label=strat, color=color, alpha=0.85)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(
        [c.replace("_", "\n") for c in class_names], fontsize=9
    )
    ax.set_ylabel("Test Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-Class Test Accuracy by Freezing Strategy")
    ax.legend(fontsize=8)
    plt.tight_layout()
    path = OUT_DIR / "per_class_breakdown.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"Saved: {path}")

def plot_summary_table(results: dict):
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.axis("off")
    rows = []
    for strat in STRATEGIES:
        r   = results[strat]
        gap = r["overfitting_gap"][-1]
        rows.append([
            strat,
            f"{r['test_acc']:.3f}",
            f"{gap:+.3f}",
            f"{r['total_training_time_s']:.0f}s",
            f"{r['avg_epoch_time_s']:.1f}s",
            str(len(r["train_accs"])),   # actual epochs run (early stop aware)
        ])
    cols = ["Strategy", "Test Acc", "Final Gap (train−val)",
            "Total Time", "Avg Epoch", "Epochs Run"]
    table = ax.table(cellText=rows, colLabels=cols,
                     cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)
    ax.set_title("Freezing Strategy Comparison — ResNet-50, IndiCraft",
                 fontsize=12, pad=20)
    plt.tight_layout()
    path = OUT_DIR / "training_summary.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

# ── CONSOLE SUMMARY ───────────────────────────────────────────────────────────
def print_summary(results: dict):
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    print(f"{'Strategy':<28} {'Test Acc':>9} {'Gap':>8} {'Time':>9} {'Epochs':>7}")
    print("-"*70)
    for s in STRATEGIES:
        r = results[s]
        print(f"{s:<28} {r['test_acc']:>9.3f} "
              f"{r['overfitting_gap'][-1]:>+8.3f} "
              f"{r['total_training_time_s']:>8.0f}s "
              f"{len(r['train_accs']):>7}")

# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    results, class_names = train_all()

    # Save raw numbers
    with open(OUT_DIR / "results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved: {OUT_DIR / 'results.json'}")

    # Plots
    plot_val_curves(results)
    plot_overfitting_gap(results)
    plot_per_class(results, class_names)
    plot_summary_table(results)

    # Console table
    print_summary(results)

Classes detected: ['anvil', 'blacksmith_hammer', 'blacksmith_tongs', 'carpentry_hammer', 'hand_planer', 'hand_saw', 'pottery_wheel', 'wood_chisel']

Strategy: Full fine-tune
  Trainable: 23,524,424 / 23,524,424 (100.0%)
  Ep  1  train=0.597  val=0.399  t=1165.4s
  Ep  2  train=0.873  val=0.433  t=1151.8s
  Ep  3  train=0.931  val=0.442  t=1051.9s
  Ep  4  train=0.946  val=0.442  t=1054.5s
  Ep  5  train=0.973  val=0.445  t=1063.6s
  ✓ Test acc: 0.457  |  Total time: 5487s

Strategy: Freeze early (L1+L2)
  Trainable: 18,010,120 / 23,524,424 (76.6%)
  Ep  1  train=0.706  val=0.401  t=692.4s
  Ep  2  train=0.908  val=0.430  t=709.6s
  Ep  3  train=0.943  val=0.428  t=694.1s
  Ep  4  train=0.969  val=0.435  t=680.1s
  Ep  5  train=0.980  val=0.438  t=698.7s
  ✓ Test acc: 0.467  |  Total time: 3475s

Strategy: Head only
  Trainable: 16,392 / 23,524,424 (0.1%)
  Ep  1  train=0.591  val=0.353  t=425.0s
  Ep  2  train=0.788  val=0.380  t=423.8s
  Ep  3  train=0.834  val=0.397  t=419.5s
  Ep  4